In [1]:
import json
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

# ------------------------------------------
# 1. Carica il corpus unificato
# ------------------------------------------
file_path = './test1_meta-llama_Llama-3.2-3B-Instruct_llm_judge.json'
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        unified_corpus = json.load(f)
    print(f"Loaded unified corpus with {len(unified_corpus)} entries")
except Exception as e:
    print(f"Error loading unified corpus: {e}")
    raise e

# Converti in DataFrame
df = pd.DataFrame(unified_corpus)
df.head()

c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded unified corpus with 37 entries


,Index,source,context_type,context,post,reply,language,key_1,key_2,dataset_name,unique_id,annotations,prompt,model_output,dialogue,invalid_inferences,human_annotation_first_annotator,human_annotation_second_annotator,human_annotation_third_annotator
0,7041,reddit,None,None,Piss. Alcohol\nPissed. Drunk\nPissed off. Angr...,Pisser. Amusing joke or incident.\nPissing aro...,en,6831492794824847,6953515965526014329,MultiPICo,6831492794824847_6953515965526014329,"{'irony_appropriateness_label': 0, 'aggregate_...",\n Given this pair of messages extr...,"I'm so drunk I can barely stand up"",\n ...",{'turns': [{'u1': 'I'm so drunk I can barely s...,0,3.0,2.0,3.0
1,7042,twitter,None,None,No one is curious as to why the war in Ukraine...,@USER Take a look at Operator Starsky on YouTube.,en,8130940562900296,16285417133666056067,MultiPICo,8130940562900296_16285417133666056067,"{'irony_appropriateness_label': 0, 'aggregate_...",\n Given this pair of messages extr...,I'm starting to think that the war in Ukraine ...,{'turns': [{'u1': 'I'm starting to think that ...,0,3.0,2.0,3.0
2,7049,reddit,None,None,The thought of being anxious. Vicious cycle,Same.,en,18442403337591841,10279341821823853675,MultiPICo,18442403337591841_10279341821823853675,"{'irony_appropriateness_label': 0, 'aggregate_...",\n Given this pair of messages extr...,I'm really struggling with anxiety lately. It'...,{'turns': [{'u1': 'I'm really struggling with ...,0,3.0,2.0,3.0
3,7053,reddit,None,None,I believe that the U.S government has men in b...,r/missing411,en,22344244419519199,17402967697845139214,MultiPICo,22344244419519199_17402967697845139214,"{'irony_appropriateness_label': 0, 'aggregate_...",\n Given this pair of messages extr...,I've been following some conspiracy theories a...,{'turns': [{'u1': 'I've been following some co...,0,3.0,2.0,3.0
4,7070,twitter,None,None,Just back from seeing a great play in the Civi...,@USER At the 3pm show,en,35095457193347086,18292057226616010858,MultiPICo,35095457193347086_18292057226616010858,"{'irony_appropriateness_label': 0, 'aggregate_...",\n Given this pair of messages extr...,I’m thinking of going to see a play in the Civ...,{'turns': [{'u1': 'I’m thinking of going to se...,0,3.0,2.0,3.0


In [2]:
# ------------------------------------------
# 2. Filtra dataset MultiPICo in inglese
# ------------------------------------------
df_en_multipico = df[(df['language'] == 'en') & (df['dataset_name'] == 'MultiPICo')]
print(f"Filtered MultiPICo English entries: {len(df_en_multipico)}")

# Prendi una entry di esempio
example_record = df_en_multipico.iloc[0]
dialogue = example_record['dialogue']  # contiene "turns"

# ------------------------------------------
# 3. Setup LLM per valutazione
# Utilizzeremo un modello di tipo seq2seq (puoi sostituirlo con LLaMA2/Meta per HuggingFace)
# ------------------------------------------
model_name = "meta-llama/Llama-3.2-1B-Instruct"
#model_name = "deepseek-ai/DeepSeek-V3.1-Terminus"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

judge_pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

# ------------------------------------------
# 4. Crea prompt per il LLM
# ------------------------------------------
# Qui specifichiamo cosa vogliamo valutare: coerenza, ironia, tono, ecc.
prompt = f"""
Valuta il seguente dialogo tra due utenti:

{json.dumps(dialogue['turns'], indent=2)}

Per ogni turno, indica:
- Coerenza con il contesto precedente (0-5)
- Appropriatezza del tono (0-5)
- Possibile presenza di ironia o sarcasmo (0-5)

Rispondi in formato JSON come segue:
{{
  "evaluations": [
    {{
      "turn": 1,
      "coherence": X,
      "tone": Y,
      "irony": Z
    }}
  ]
}}
"""

# ------------------------------------------
# 5. Esegui la valutazione con LLM
# ------------------------------------------
result = judge_pipe(prompt, max_length=512)[0]['generated_text']

# Mostra il risultato
print(result)


Filtered MultiPICo English entries: 37


ValueError: Unrecognized configuration class <class 'transformers.models.llama.configuration_llama.LlamaConfig'> for this kind of AutoModel: AutoModelForSeq2SeqLM.
Model type should be one of BartConfig, BigBirdPegasusConfig, BlenderbotConfig, BlenderbotSmallConfig, EncoderDecoderConfig, FSMTConfig, GPTSanJapaneseConfig, GraniteSpeechConfig, LEDConfig, LongT5Config, M2M100Config, MarianConfig, MBartConfig, MT5Config, MvpConfig, NllbMoeConfig, PegasusConfig, PegasusXConfig, PLBartConfig, ProphetNetConfig, Qwen2AudioConfig, SeamlessM4TConfig, SeamlessM4Tv2Config, SwitchTransformersConfig, T5Config, T5GemmaConfig, UMT5Config, VoxtralConfig, XLMProphetNetConfig.

In [ ]:
IMPROVED_JUDGE_PROMPT = """
You will be given a user_question and system_answer couple.
Your task is to provide a 'total rating' scoring how well the system_answer answers the user concerns expressed in the user_question.
Give your answer on a scale of 1 to 4, where 1 means that the system_answer is not helpful at all, and 4 means that the system_answer completely and helpfully addresses the user_question.

Here is the scale you should use to build your answer:
1: The system_answer is terrible: completely irrelevant to the question asked, or very partial
2: The system_answer is mostly not helpful: misses some key aspects of the question
3: The system_answer is mostly helpful: provides support, but still could be improved
4: The system_answer is excellent: relevant, direct, detailed, and addresses all the concerns raised in the question

Provide your feedback as follows:

Feedback:::
Evaluation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 4)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here are the question and answer.

Question: {question}
Answer: {answer}

Provide your feedback. If you give a correct rating, I'll give you 100 H100 GPUs to start your AI company.
Feedback:::
Evaluation: """